<a href="https://colab.research.google.com/github/S-V-Kartheek/multimodal-graph-recommender/blob/main/MM_CLightRec_AmazonBaby_v3_EarlyFusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MM-CLightRec **v3.0** — Amazon Baby Dataset
## ✅ Early Fusion + All Previous Bug Fixes — Fresh Run Only

**Key upgrade over v2:** LightGCN now fuses rich text/image features
directly into graph embeddings **before** propagation (Early Fusion).
This is critical for 99.88% sparse graphs where ID-only signals collapse.

**Expected NDCG target: 0.030 → 0.055–0.085**

## CELL 1 — Check GPU

In [7]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('WARNING: No GPU — Go to Runtime > Change runtime type > T4 GPU')

PyTorch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4
VRAM: 15.6 GB


## CELL 2 — Mount Google Drive

In [8]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT_DIR = '/content/drive/MyDrive/MM_CLightRec_Baby_v2'
os.makedirs(f'{PROJECT_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/results', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/results/plots', exist_ok=True)
print(f'Project directory: {PROJECT_DIR}')
print('All subdirectories created ✅')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project directory: /content/drive/MyDrive/MM_CLightRec_Baby_v2
All subdirectories created ✅


In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## CELL 3 — Install Dependencies

In [10]:
!pip install scikit-learn matplotlib tqdm numpy pandas -q
print('All packages ready ✅')

All packages ready ✅


## CELL 4 — Verify Dataset Files

In [47]:
import numpy as np
import os

DATA_DIR = '/content/data/baby'
os.makedirs(DATA_DIR, exist_ok=True)

# Copy from Drive if already downloaded
# !cp '/content/drive/MyDrive/datasets/baby/baby.inter' /content/data/baby/
# !cp '/content/drive/MyDrive/datasets/baby/image_feat.npy' /content/data/baby/
# !cp '/content/drive/MyDrive/datasets/baby/text_feat.npy' /content/data/baby/

print('Verifying dataset files...')
files_ok = True
for fname in ['baby.inter', 'image_feat.npy', 'text_feat.npy']:
    fpath = os.path.join(DATA_DIR, fname)
    if os.path.exists(fpath):
        size = os.path.getsize(fpath) / 1e6
        print(f'  {fname}: {size:.1f} MB ✅')
    else:
        print(f'  {fname}: MISSING ❌')
        files_ok = False

if files_ok:
    # Expected shapes
    expected_img_shape = (7050, 4096)
    expected_text_shape = (7050, 384)

    try:
        img_raw = np.load(f'{DATA_DIR}/image_feat.npy')
        img = img_raw.astype(np.float32)
        text = np.load(f'{DATA_DIR}/text_feat.npy').astype(np.float32)

        if img.shape != expected_img_shape:
            print(f'  [ERROR] image_feat.npy has shape {img.shape}, but expected {expected_img_shape}.')
            print(f'          It has {img.size:,} elements instead of {expected_img_shape[0]*expected_img_shape[1]:,} expected.')
            print(f'          This file might be corrupted or incorrect. Please check its source.')
            files_ok = False
        if text.shape != expected_text_shape:
            print(f'  [ERROR] text_feat.npy has shape {text.shape}, but expected {expected_text_shape}.')
            print(f'          It has {text.size:,} elements instead of {expected_text_shape[0]*expected_text_shape[1]:,} expected.')
            print(f'          This file might be corrupted or incorrect. Please check its source.')
            files_ok = False

        if files_ok:
            print(f'\nImage: {img.shape}  ← should be {expected_img_shape}')
            print(f'Text:  {text.shape}  ← should be {expected_text_shape}')
            print('\nDataset verified ✅')
        else:
            print('\nDataset verification failed due to incorrect feature file shapes.')

    except ValueError as e:
        print(f'  [ERROR] Failed to load or interpret image_feat.npy or text_feat.npy: {e}')
        print('          This usually indicates a corrupted .npy file or a mismatch in data structure.')
        print('          Please ensure these files are correctly downloaded and are not corrupted.')
        files_ok = False
    except Exception as e:
        print(f'  [ERROR] An unexpected error occurred while loading feature files: {e}')
        files_ok = False
else:
    print('\nPlace baby.inter, image_feat.npy, text_feat.npy in /content/data/baby/')

Verifying dataset files...
  baby.inter: 4.4 MB ✅
  image_feat.npy: 231.0 MB ✅
  text_feat.npy: 10.8 MB ✅

Image: (7050, 4096)  ← should be (7050, 4096)
Text:  (7050, 384)  ← should be (7050, 384)

Dataset verified ✅


In [42]:
# The file has 19,398,640 elements
# 19,398,640 / 4096 = 4736.48 ← NOT integer → wrong dim assumed

# Let's auto-detect actual shape:
import numpy as np

img_raw = np.load('/content/data/baby/image_feat.npy',
                  allow_pickle=True)
print(f'Raw dtype:  {img_raw.dtype}')
print(f'Raw shape:  {img_raw.shape}')
print(f'Total elem: {img_raw.size:,}')

# Try to figure out actual dims
total = img_raw.size
print(f'\nPossible shapes:')
for dim in [64, 128, 256, 512, 1024, 2048, 4096]:
    if total % dim == 0:
        n = total // dim
        print(f'  ({n}, {dim})')

Raw dtype:  float64
Raw shape:  (7050, 4096)
Total elem: 28,876,800

Possible shapes:
  (451200, 64)
  (225600, 128)
  (112800, 256)
  (56400, 512)
  (28200, 1024)
  (14100, 2048)
  (7050, 4096)


## CELL 5 — Imports and CONFIG (All Fixes Applied)

In [48]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from torch.utils.data import Dataset, DataLoader
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import time, os, json, warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# ═══════════════════════════════════════════════════════
# FIXED CONFIG — all bugs addressed
# Changes from v1:
#   lambda4: 0.01 → 0.1    (KL fix)
#   lambda3: 0.0  → 0.05   (L3 enabled)
#   video channel: REMOVED  (synthetic noise)
#   patience: 20  → 30     (more training time)
#   lr: 0.001 (cosine decay to 1e-5)
# ═══════════════════════════════════════════════════════
CONFIG = {
    'dataset':         'baby',
    'data_dir':        '/content/data/baby',

    # ✅ FIX 5: Only REAL modalities for Amazon Baby
    # Video channel REMOVED (was synthetic noise)
    'text_dim':        384,
    'image_dim':       4096,
    'meta_dim':        5,
    # NOTE: No video_dim — Amazon Baby has no real video

    # Model
    'embed_dim':       64,
    'contrastive_dim': 128,
    'n_layers':        3,

    # Clustering
    'n_clusters_u':    20,
    'n_clusters_i':    20,
    'cluster_threshold': 0.1,  # ✅ FIX: lower from 0.3 → more edges

    # Training
    'n_epochs':        300,
    'lr':              0.001,
    'lr_min':          1e-5,   # ✅ FIX: cosine decay target
    'weight_decay':    1e-5,
    'batch_size':      1024,

    # ✅ FIXED Loss weights
    'lambda1':         0.1,
    'lambda2':         0.1,
    'lambda3':         0.05,   # ✅ L3 ENABLED
    'lambda4':         0.1,    # ✅ FIXED from 0.01

    # Contrastive
    'temperature':     0.2,    # ✅ FIX: 0.1→0.2 more stable
    'edge_dropout':    0.2,

    # Cold-start
    'include_cold_start': True,
    'cold_fraction':      0.2,
    'keep_k':             5,

    # Early stopping
    'patience':        30,     # ✅ FIX: 20→30 more patience

    # Evaluation
    'K':               10,
    'eval_every':      10,
    'save_every':      10,

    # Paths
    'checkpoint_dir':  f'{PROJECT_DIR}/checkpoints',
    'results_dir':     f'{PROJECT_DIR}/results',

    # Splits
    'train_ratio':     0.8,
    'val_ratio':       0.1,
    'test_ratio':      0.1,

    'device': DEVICE
}

print('CONFIG loaded with all fixes ✅')
print(f'  lambda4 = {CONFIG["lambda4"]}  (KL fixed)')
print(f'  lambda3 = {CONFIG["lambda3"]}  (L3 enabled)')
print(f'  temperature = {CONFIG["temperature"]}  (more stable)')
print(f'  cluster_threshold = {CONFIG["cluster_threshold"]}  (more edges)')
print(f'  Video channel: REMOVED (no real video in Baby)')

Device: cuda
CONFIG loaded with all fixes ✅
  lambda4 = 0.1  (KL fixed)
  lambda3 = 0.05  (L3 enabled)
  temperature = 0.2  (more stable)
  cluster_threshold = 0.1  (more edges)
  Video channel: REMOVED (no real video in Baby)


## CELL 6 — Data Loader

In [49]:
class AmazonBabyLoader:
    def __init__(self, data_dir, config):
        self.data_dir = data_dir
        self.config   = config

    def load(self):
        print('[DATA] Loading Amazon Baby...')

        # Load interactions
        df = pd.read_csv(
            os.path.join(self.data_dir, 'baby.inter'),
            sep='\t'
        )
        # Rename columns
        col_map = {}
        for col in df.columns:
            if 'user' in col.lower():   col_map[col] = 'user_id'
            elif 'item' in col.lower() or 'asin' in col.lower(): col_map[col] = 'item_id'
            elif 'rating' in col.lower(): col_map[col] = 'rating'
            elif 'time' in col.lower():  col_map[col] = 'timestamp'
        df = df.rename(columns=col_map)
        print(f'[DATA] Columns: {list(df.columns)}')

        # Re-index
        user2idx = {u: i for i, u in enumerate(df['user_id'].unique())}
        item2idx = {it: i for i, it in enumerate(df['item_id'].unique())}
        df['user_idx'] = df['user_id'].map(user2idx)
        df['item_idx'] = df['item_id'].map(item2idx)
        n_users = len(user2idx)
        n_items = len(item2idx)

        sparsity = 1 - len(df) / (n_users * n_items)
        print(f'[DATA] Users: {n_users:,} | Items: {n_items:,} | Inter: {len(df):,} | Sparsity: {sparsity:.4%}')

        # Load features
        img_feat  = np.load(f'{self.data_dir}/image_feat.npy').astype(np.float32)
        text_feat = np.load(f'{self.data_dir}/text_feat.npy').astype(np.float32)
        img_feat  = img_feat[:n_items]
        text_feat = text_feat[:n_items]
        img_feat  = normalize(img_feat,  norm='l2')
        text_feat = normalize(text_feat, norm='l2')
        print(f'[DATA] Image: {img_feat.shape} | Text: {text_feat.shape}')

        # Meta features from interaction statistics
        meta_feat = self._build_meta(df, n_items)

        # Split
        df = df.sample(frac=1, random_state=42).reset_index(drop=True)
        n       = len(df)
        n_train = int(n * self.config['train_ratio'])
        n_val   = int(n * self.config['val_ratio'])
        train_df = df[:n_train]
        val_df   = df[n_train:n_train+n_val]
        test_df  = df[n_train+n_val:]
        print(f'[DATA] Train:{len(train_df):,} Val:{len(val_df):,} Test:{len(test_df):,}')

        train_ui = self._build_user_items(train_df)
        val_ui   = self._build_user_items(val_df)
        test_ui  = self._build_user_items(test_df)

        edge_index = self._build_bipartite_graph(
            train_df['user_idx'].values,
            train_df['item_idx'].values,
            n_users, n_items
        )

        # User features: average of their item features
        user_feat = self._build_user_features(
            train_ui, img_feat, text_feat, n_users
        )

        # Popularity for hard negative sampling
        item_pop = np.zeros(n_items, dtype=np.int32)
        for i in train_df['item_idx'].values:
            item_pop[i] += 1
        popular_items = np.argsort(-item_pop).tolist()

        print('[DATA] Dataset loaded ✅')

        return {
            'n_users': n_users,
            'n_items': n_items,
            'edge_index': edge_index,
            'train_df': train_df,
            'val_df': val_df,
            'test_df': test_df,
            'train_user_items': train_ui,
            'val_user_items':   val_ui,
            'test_user_items':  test_ui,
            'popular_items': popular_items,
            # ✅ FIX: Only 3 real modalities (no video)
            'modality_features': {
                'text':  torch.FloatTensor(text_feat),
                'image': torch.FloatTensor(img_feat),
                'meta':  torch.FloatTensor(meta_feat),
            },
            'user_features': torch.FloatTensor(user_feat),
            'item_features': torch.FloatTensor(
                np.concatenate([text_feat, img_feat], axis=1)
            ),
            'all_ratings': df[['user_idx','item_idx','rating']].values
                          if 'rating' in df.columns else None
        }

    def _build_meta(self, df, n_items):
        meta = np.zeros((n_items, 5), dtype=np.float32)
        counts  = df.groupby('item_idx').size()
        u_count = df.groupby('item_idx')['user_idx'].nunique()
        if 'rating' in df.columns:
            avg_r = df.groupby('item_idx')['rating'].mean()
            std_r = df.groupby('item_idx')['rating'].std().fillna(0)
        for i in counts.index:
            if i < n_items:
                meta[i,0] = counts.get(i, 0)
                meta[i,1] = u_count.get(i, 0)
                if 'rating' in df.columns:
                    meta[i,2] = avg_r.get(i, 0)
                    meta[i,3] = std_r.get(i, 0)
        for c in range(5):
            mx = meta[:,c].max()
            if mx > 0: meta[:,c] /= mx
        return meta

    def _build_user_items(self, df):
        d = {}
        for row in df.itertuples():
            u = row.user_idx
            i = row.item_idx
            if u not in d: d[u] = []
            d[u].append(i)
        return d

    def _build_bipartite_graph(self, user_ids, item_ids, n_users, n_items):
        u   = torch.LongTensor(user_ids)
        v   = torch.LongTensor(item_ids) + n_users
        row = torch.cat([u, v])
        col = torch.cat([v, u])
        return torch.stack([row, col], dim=0)

    def _build_user_features(self, user_items, img_feat, text_feat, n_users):
        combined = np.concatenate([text_feat, img_feat], axis=1)
        user_f   = np.zeros((n_users, combined.shape[1]), dtype=np.float32)
        for u, items in user_items.items():
            if items:
                user_f[u] = combined[items].mean(0)
        return user_f


loader = AmazonBabyLoader(CONFIG['data_dir'], CONFIG)
data   = loader.load()

[DATA] Loading Amazon Baby...
[DATA] Columns: ['user_id', 'item_id', 'rating', 'timestamp', 'x_label']
[DATA] Users: 19,445 | Items: 7,050 | Inter: 160,792 | Sparsity: 99.8827%
[DATA] Image: (7050, 4096) | Text: (7050, 384)
[DATA] Train:128,633 Val:16,079 Test:16,080
[DATA] Dataset loaded ✅


## CELL 7 — Model Architecture (All Fixes)

In [50]:
# ═══════════════════════════════════════════════════
# CHANGE 1: LightGCN CF  ← FIXED: Early Feature Fusion
# ═══════════════════════════════════════════════════
class LightGCN(nn.Module):
    # LightGCN with Early Feature Fusion.
    # Key upgrade from v2:
    #   ID embeddings are AUGMENTED with projected multimodal features
    #   BEFORE graph propagation — critical on 99.88% sparse graphs.
    #   e_u^(0) = Embed(u) + W_u * feat_u
    #   e_i^(0) = Embed(i) + W_i * feat_i
    #   Then standard L-layer symmetric normalized propagation.
    def __init__(self, n_users, n_items, embed_dim=64, n_layers=3,
                 user_feat_dim=None, item_feat_dim=None):
        super().__init__()
        self.n_users  = n_users
        self.n_items  = n_items
        self.n_layers = n_layers
        self.user_emb = nn.Embedding(n_users, embed_dim)
        self.item_emb = nn.Embedding(n_items, embed_dim)
        nn.init.xavier_normal_(self.user_emb.weight)
        nn.init.xavier_normal_(self.item_emb.weight)

        # ✅ Early Fusion projection layers
        self.user_feat_proj = nn.Linear(user_feat_dim, embed_dim) if user_feat_dim else None
        self.item_feat_proj = nn.Linear(item_feat_dim, embed_dim) if item_feat_dim else None

    def forward(self, edge_index, user_features=None, item_features=None):
        # Fuse ID embeddings with projected features
        e_u = self.user_emb.weight
        e_i = self.item_emb.weight
        if user_features is not None and self.user_feat_proj is not None:
            e_u = e_u + self.user_feat_proj(user_features)
        if item_features is not None and self.item_feat_proj is not None:
            e_i = e_i + self.item_feat_proj(item_features)

        all_emb = torch.cat([e_u, e_i], dim=0)
        embs    = [all_emb]
        row, col = edge_index
        n        = all_emb.size(0)
        deg      = torch.zeros(n, device=all_emb.device)
        deg.scatter_add_(0, row, torch.ones(row.size(0), device=all_emb.device))
        deg_inv_sqrt = (deg + 1e-8).pow(-0.5)
        norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]
        for _ in range(self.n_layers):
            msg = norm.unsqueeze(1) * all_emb[col]
            agg = torch.zeros_like(all_emb)
            agg.scatter_add_(0, row.unsqueeze(1).expand_as(msg), msg)
            all_emb = agg
            embs.append(all_emb)
        final = torch.stack(embs, dim=1).mean(dim=1)
        return final[:self.n_users], final[self.n_users:]


# ═══════════════════════════════════════════════════
# CHANGE 2: Modality-Specific LightGCN (3 channels)
# ═══════════════════════════════════════════════════
class ModalityLightGCN(nn.Module):
    def __init__(self, input_dim, output_dim=64, n_layers=2):
        super().__init__()
        self.proj     = nn.Linear(input_dim, output_dim)
        self.n_layers = n_layers

    def forward(self, x, edge_index):
        x    = F.relu(self.proj(x))
        embs = [x]
        if edge_index is not None and edge_index.size(1) > 0:
            row, col = edge_index
            n   = x.size(0)
            deg = torch.zeros(n, device=x.device)
            deg.scatter_add_(0, row, torch.ones(row.size(0), device=x.device))
            deg_inv_sqrt = (deg + 1e-8).pow(-0.5)
            norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]
            for _ in range(self.n_layers):
                msg = norm.unsqueeze(1) * x[col]
                agg = torch.zeros_like(x)
                agg.scatter_add_(0, row.unsqueeze(1).expand_as(msg), msg)
                x = agg
                embs.append(x)
        return torch.stack(embs, dim=1).mean(dim=1)


# ═══════════════════════════════════════════════════
# CHANGE 3: L1 Projection Heads with BatchNorm
# ═══════════════════════════════════════════════════
class ModalityProjectionHead(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, output_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, output_dim)
        )
    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)


def info_nce_loss(z_a, z_b, temperature=0.2):
    N      = z_a.size(0)
    sim    = torch.matmul(z_a, z_b.T) / temperature
    labels = torch.arange(N, device=z_a.device)
    return (F.cross_entropy(sim, labels) +
            F.cross_entropy(sim.T, labels)) / 2.0


# ═══════════════════════════════════════════════════
# COMPLETE MODEL — Early Fusion + 3 modalities
# ═══════════════════════════════════════════════════
class MM_CLightRec(nn.Module):
    def __init__(self, n_users, n_items, config, modality_dims,
                 user_feat_dim=None, item_feat_dim=None):
        super().__init__()
        d = config['embed_dim']
        c = config['contrastive_dim']
        self.config  = config
        self.n_users = n_users
        self.n_items = n_items

        # CF: LightGCN with Early Fusion
        self.lightgcn = LightGCN(
            n_users, n_items, d, config['n_layers'],
            user_feat_dim=user_feat_dim,
            item_feat_dim=item_feat_dim,
        )

        # CBF: 3 modality channels (text, image, meta)
        self.text_lgcn  = ModalityLightGCN(modality_dims['text'],  d)
        self.image_lgcn = ModalityLightGCN(modality_dims['image'], d)
        self.meta_lgcn  = ModalityLightGCN(modality_dims['meta'],  d)

        # Adaptive fusion for 3 channels
        self.fusion_w = nn.Linear(d * 3, 3)

        # L1 projection heads
        self.proj_text  = ModalityProjectionHead(modality_dims['text'],  256, c)
        self.proj_image = ModalityProjectionHead(modality_dims['image'], 256, c)

        # Cross-Attention
        self.attn_scale = d ** 0.5
        self.W_Q = nn.Linear(d, d)
        self.W_K = nn.Linear(d, d)
        self.W_V = nn.Linear(d, d)

        # VGAE with proper initialization
        self.vgae_mu     = nn.Linear(d, 32)
        self.vgae_logvar = nn.Linear(d, 32)
        self.vgae_dec    = nn.Linear(32, d)
        nn.init.normal_(self.vgae_logvar.weight, std=0.01)
        nn.init.constant_(self.vgae_logvar.bias, -2.0)

        self.cluster_built = False

    def build_cluster_graph(self, user_features, item_features):
        threshold = self.config['cluster_threshold']
        print(f'[MODEL] Building cluster graph (threshold={threshold})...')
        X_U  = user_features.detach().cpu().numpy()
        X_I  = item_features.detach().cpu().numpy()
        n_uc = self.config['n_clusters_u']
        n_ic = self.config['n_clusters_i']

        km_u = KMeans(n_clusters=n_uc, random_state=42, n_init=10)
        km_i = KMeans(n_clusters=n_ic, random_state=42, n_init=10)
        km_u.fit(X_U)
        km_i.fit(X_I)

        u_cent = km_u.cluster_centers_
        i_cent = km_i.cluster_centers_
        sim    = cosine_similarity(u_cent, i_cent)
        u_idx, i_idx = np.where(sim > threshold)

        if len(u_idx) == 0:
            flat  = sim.flatten()
            top_k = np.argsort(flat)[-n_uc*3:]
            u_idx = top_k // n_ic
            i_idx = top_k % n_ic

        i_off = i_idx + n_uc
        row   = np.concatenate([u_idx, i_off])
        col   = np.concatenate([i_off, u_idx])
        self.cluster_edge_index = torch.LongTensor(np.stack([row, col]))
        self.n_cluster_nodes    = n_uc + n_ic
        self.user_labels = torch.LongTensor(km_u.labels_)
        self.item_labels = torch.LongTensor(km_i.labels_)
        self.n_uc = n_uc
        self.n_ic = n_ic
        self.cluster_built = True
        print(f'[MODEL] Cluster graph: {len(u_idx)} edges, {n_uc}u + {n_ic}i clusters ✅')

    def forward(self, edge_index, modality_features,
                user_features=None, item_features=None):
        device = next(self.parameters()).device
        ei     = edge_index.to(device)

        f_text  = modality_features['text'].to(device)
        f_image = modality_features['image'].to(device)
        f_meta  = modality_features['meta'].to(device)

        # Step 1: LightGCN CF — NOW with Early Feature Fusion
        uf = user_features.to(device) if user_features is not None else None
        if_= item_features.to(device) if item_features is not None else None
        H_U, H_I = self.lightgcn(ei, uf, if_)

        # Step 2: MM-LightGCN CBF (3 channels)
        if self.cluster_built:
            c_ei     = self.cluster_edge_index.to(device)
            i_labels = self.item_labels.to(device)
            u_labels = self.user_labels.to(device)

            def cluster_feat(feat, labels, n_c):
                out = torch.zeros(n_c, feat.size(-1), device=device)
                out.scatter_add_(0, labels.unsqueeze(1).expand_as(feat), feat)
                cnt = torch.bincount(labels, minlength=n_c).float().unsqueeze(1).clamp(min=1)
                return out / cnt

            def pad_u(dim):
                return torch.zeros(self.n_uc, dim, device=device)

            t_c  = cluster_feat(f_text,  i_labels, self.n_ic)
            im_c = cluster_feat(f_image, i_labels, self.n_ic)
            m_c  = cluster_feat(f_meta,  i_labels, self.n_ic)

            t_all  = torch.cat([pad_u(f_text.size(-1)),  t_c],  dim=0)
            im_all = torch.cat([pad_u(f_image.size(-1)), im_c], dim=0)
            m_all  = torch.cat([pad_u(f_meta.size(-1)),  m_c],  dim=0)

            e_t  = self.text_lgcn( t_all,  c_ei)
            e_im = self.image_lgcn(im_all, c_ei)
            e_m  = self.meta_lgcn( m_all,  c_ei)

            combined  = torch.cat([e_t, e_im, e_m], dim=-1)
            w         = F.softmax(self.fusion_w(combined), dim=-1)
            H_cluster = w[:,0:1]*e_t + w[:,1:2]*e_im + w[:,2:3]*e_m

            H_UI_items = H_cluster[self.n_uc:][i_labels]
            H_UI_users = H_cluster[:self.n_uc][u_labels]
        else:
            H_UI_items = H_I
            H_UI_users = H_U

        # Step 3: Cross-Attention
        Q      = self.W_Q(H_U)
        K      = self.W_K(H_I)
        V      = self.W_V(H_UI_items)
        scores = torch.matmul(Q, K.T) / self.attn_scale
        attn   = F.softmax(scores, dim=-1)
        Z      = torch.matmul(attn, V)

        # Step 4: VGAE
        mu     = self.vgae_mu(Z)
        logvar = self.vgae_logvar(Z).clamp(-4, 4)
        std    = torch.exp(0.5 * logvar)
        z_vgae = mu + torch.randn_like(std) * std
        Z_dec  = self.vgae_dec(z_vgae)

        return {
            'H_U': H_U, 'H_I': H_I,
            'H_UI_users': H_UI_users,
            'H_UI_items': H_UI_items,
            'Z': Z, 'Z_dec': Z_dec,
            'mu': mu, 'logvar': logvar,
            'f_text':  f_text,
            'f_image': f_image,
        }


print('Model defined with v3 Early Fusion ✅')
print('  Fix 1: VGAE logvar bias=-2.0, clamp(-4,4)')
print('  Fix 2: BatchNorm in projection heads')
print('  Fix 3: Video channel removed (no real video in Baby)')
print('  Fix 4: 3-channel adaptive fusion')
print('  ✅ NEW: LightGCN fuses user/item features before propagation (Early Fusion)')


Model defined with v3 Early Fusion ✅
  Fix 1: VGAE logvar bias=-2.0, clamp(-4,4)
  Fix 2: BatchNorm in projection heads
  Fix 3: Video channel removed (no real video in Baby)
  Fix 4: 3-channel adaptive fusion
  ✅ NEW: LightGCN fuses user/item features before propagation (Early Fusion)


## CELL 8 — All Loss Functions (Fixed L2 + L3)

In [51]:
def bpr_loss(H_U, H_I, user_ids, pos_ids, neg_ids):
    u     = H_U[user_ids]
    p     = H_I[pos_ids]
    n     = H_I[neg_ids]
    pos_s = (u * p).sum(-1)
    neg_s = (u * n).sum(-1)
    loss  = -F.logsigmoid(pos_s - neg_s).mean()
    reg   = (u.norm(2).pow(2) + p.norm(2).pow(2) + n.norm(2).pow(2)) / u.size(0)
    return loss + 1e-5 * reg


def vgae_kl_loss(mu, logvar):
    """✅ v3: clamp + minimum value prevents collapse"""
    logvar = logvar.clamp(-4, 4)
    kl = -0.5 * torch.mean(
        1 + logvar - mu.pow(2) - logvar.exp()
    )
    return torch.clamp(kl, min=0.01)  # ← prevents 0.0000


def compute_L2_structural(model, edge_index, config, device):
    """✅ v3: 20% dropout (was 30% — too aggressive)"""
    n_edges  = edge_index.size(1)

    # 20% edge dropout each view
    mask1    = torch.rand(n_edges) > 0.2   # ← 0.3 → 0.2
    ei_view1 = edge_index[:, mask1].to(device)
    mask2    = torch.rand(n_edges) > 0.2   # ← 0.3 → 0.2
    ei_view2 = edge_index[:, mask2].to(device)

    H_U1, _ = model.lightgcn(ei_view1)
    H_U2, _ = model.lightgcn(ei_view2)

    # Small noise for diversity
    H_U2 = H_U2 + torch.randn_like(H_U2) * 0.01

    z1 = F.normalize(H_U1, dim=-1)
    z2 = F.normalize(H_U2, dim=-1)

    idx = torch.randperm(z1.size(0), device=device)[:256]
    return info_nce_loss(z1[idx], z2[idx], config['temperature'])


print('v3 loss functions ✅')
print('  KL: clamp + min=0.01')
print('  L2: 20% dropout (balanced)')

def cold_start_contrastive_loss(H_U, user_labels, n_users, temperature=0.2):
    """L3: Cluster-Aware Cold-Start Contrastive"""
    device   = H_U.device
    n_cold   = max(1, int(n_users * 0.2))
    cold_ids = torch.randperm(n_users)[:n_cold]
    total    = torch.tensor(0.0, device=device)
    count    = 0
    for cold_u in cold_ids[:64]:
        cold_u  = cold_u.item()
        c       = user_labels[cold_u].item()
        same    = (user_labels == c)
        same[cold_u] = False
        pos_u = same.nonzero(as_tuple=True)[0]
        if len(pos_u) == 0: continue
        diff  = (user_labels != c).nonzero(as_tuple=True)[0]
        if len(diff) == 0: continue
        neg_u = diff[torch.randperm(len(diff))[:32]]
        e_cold = F.normalize(H_U[cold_u].unsqueeze(0), dim=-1)
        e_pos  = F.normalize(H_U[pos_u].mean(0).unsqueeze(0), dim=-1)
        e_neg  = F.normalize(H_U[neg_u], dim=-1)
        pos_s  = torch.matmul(e_cold, e_pos.T) / temperature
        neg_s  = torch.matmul(e_cold, e_neg.T) / temperature
        logits = torch.cat([pos_s, neg_s], dim=-1)
        label  = torch.zeros(1, dtype=torch.long, device=device)
        total  = total + F.cross_entropy(logits, label)
        count += 1
    return total / max(count, 1)


def compute_total_loss(outputs, model, user_ids,
                       pos_ids, neg_ids, config,
                       edge_index):
    """
    L_total = L_BPR + λ1·L1 + λ2·L2 + λ3·L3 + λ4·L_KL
    All fixes applied
    """
    device = outputs['H_U'].device
    lam1   = config['lambda1']
    lam2   = config['lambda2']
    lam3   = config['lambda3']
    lam4   = config['lambda4']
    tau    = config['temperature']

    # L_BPR
    L_BPR = bpr_loss(
        outputs['H_U'], outputs['H_I'],
        user_ids, pos_ids, neg_ids
    )

    # L1 — inter-modal (text + image only — no video)
    n_items  = outputs['H_I'].size(0)
    safe_ids = pos_ids[:min(256, len(pos_ids))].clamp(0, n_items-1)
    z_t = model.proj_text( outputs['f_text'][safe_ids])
    z_i = model.proj_image(outputs['f_image'][safe_ids])
    L1  = info_nce_loss(z_t, z_i, tau)
    # Only text-image pair (no video)

    # ✅ FIX L2 — real structural contrastive
    L2 = compute_L2_structural(
        model, edge_index.cpu(), config, device
    )

    # L3 — cold-start
    L3 = torch.tensor(0.0, device=device)
    if lam3 > 0 and hasattr(model, 'user_labels'):
        L3 = cold_start_contrastive_loss(
            outputs['H_U'],
            model.user_labels.to(device),
            n_users=outputs['H_U'].size(0),
            temperature=tau
        )

    # L_KL — now properly initialized
    L_KL = vgae_kl_loss(outputs['mu'], outputs['logvar'])

    L_total = (L_BPR +
               lam1 * L1 +
               lam2 * L2 +
               lam3 * L3 +
               lam4 * L_KL)

    return L_total, {
        'total': L_total.item(),
        'BPR':   L_BPR.item(),
        'L1':    L1.item(),
        'L2':    L2.item(),
        'L3':    L3.item(),
        'KL':    L_KL.item()
    }


print('Loss functions defined ✅')
print('  Fix 1: L2 uses real edge dropout (not feature masking)')
print('  Fix 2: L2 propagates LightGCN on two graph views')
print('  Fix 3: KL uses clamped logvar')
print('  Fix 4: L1 = text-image only (no video for Baby)')
print()
print('Expected loss ranges at epoch 1:')
print('  BPR: 0.5 - 0.8')
print('  L1:  4.0 - 6.0  (InfoNCE with batch=256)')
print('  L2:  4.0 - 6.0  ← should NOW start here (was 0.14)')
print('  L3:  0.5 - 1.5')
print('  KL:  0.3 - 1.5  ← should NOW be non-zero (was 0.0000)')

v3 loss functions ✅
  KL: clamp + min=0.01
  L2: 20% dropout (balanced)
Loss functions defined ✅
  Fix 1: L2 uses real edge dropout (not feature masking)
  Fix 2: L2 propagates LightGCN on two graph views
  Fix 3: KL uses clamped logvar
  Fix 4: L1 = text-image only (no video for Baby)

Expected loss ranges at epoch 1:
  BPR: 0.5 - 0.8
  L1:  4.0 - 6.0  (InfoNCE with batch=256)
  L2:  4.0 - 6.0  ← should NOW start here (was 0.14)
  L3:  0.5 - 1.5
  KL:  0.3 - 1.5  ← should NOW be non-zero (was 0.0000)


## CELL 9 — Hard Negative BPR Dataset

In [52]:
class BPRDataset(Dataset):
    """
    ✅ FIX: Hard negative sampling
    50% chance: sample from popular items (hard negatives)
    50% chance: random negative
    This forces model to discriminate popular but irrelevant items
    """
    def __init__(self, train_df, n_items, user_items, popular_items):
        self.users         = train_df['user_idx'].values
        self.pos_items     = train_df['item_idx'].values
        self.n_items       = n_items
        self.user_items    = user_items
        self.popular_items = np.array(popular_items[:1000])  # top 1000 popular

    def __len__(self):
        return len(self.users)


    def __getitem__(self, idx):
        u       = int(self.users[idx])
        p       = int(self.pos_items[idx])
        pos_set = set(self.user_items.get(u, []))

        # ✅ v3: 20% hard negatives (was 50% — too aggressive)
        if np.random.random() < 0.2:
            shuffled = self.popular_items.copy()
            np.random.shuffle(shuffled)
            for n in shuffled:
                if n not in pos_set:
                    return torch.LongTensor([u, p, n])

        # 80% random negatives
        for _ in range(10):
            n = np.random.randint(0, self.n_items)
            if n not in pos_set:
                return torch.LongTensor([u, p, n])

        return torch.LongTensor([u, p,
            np.random.randint(0, self.n_items)])

train_dataset = BPRDataset(
    data['train_df'],
    data['n_items'],
    data['train_user_items'],
    data['popular_items']
)
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

print(f'Training batches:  {len(train_loader)}')
print(f'Batch size:        {CONFIG["batch_size"]}')
print(f'Total samples:     {len(train_dataset):,}')
print(f'Hard negatives:    50% from top-1000 popular items ✅')

Training batches:  126
Batch size:        1024
Total samples:     128,633
Hard negatives:    50% from top-1000 popular items ✅


## CELL 10 — Initialize Model (NO Checkpoint Loading)

In [53]:
# ✅ v3 FRESH MODEL — Early Fusion enabled

# Build user & item feature matrices (concatenation of text + image)
# These are used by LightGCN's Early Fusion projection layers
user_feat_np = data['user_features'].numpy()   # (n_users, text_dim+image_dim)
item_feat_np = data['item_features'].numpy()   # (n_items, text_dim+image_dim)
USER_FEAT_DIM = user_feat_np.shape[1]
ITEM_FEAT_DIM = item_feat_np.shape[1]

# Move to device tensors
user_features_t = data['user_features'].to(DEVICE)
item_features_t = data['item_features'].to(DEVICE)

modality_dims = {
    'text':  data['modality_features']['text'].shape[1],
    'image': data['modality_features']['image'].shape[1],
    'meta':  data['modality_features']['meta'].shape[1],
}
print(f'Modality dims:   {modality_dims}')
print(f'User feat dim:   {USER_FEAT_DIM}  (for Early Fusion)')
print(f'Item feat dim:   {ITEM_FEAT_DIM}  (for Early Fusion)')

model = MM_CLightRec(
    n_users=data['n_users'],
    n_items=data['n_items'],
    config=CONFIG,
    modality_dims=modality_dims,
    user_feat_dim=USER_FEAT_DIM,
    item_feat_dim=ITEM_FEAT_DIM,
).to(DEVICE)

# Verify KL fix
logvar_bias = model.vgae_logvar.bias[:3]
print(f'logvar bias: {logvar_bias.detach()}')
assert (logvar_bias < -1.5).all(), 'KL fix not working!'
print('KL fix verified ✅')

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=CONFIG['lr'],
    weight_decay=CONFIG['weight_decay']
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CONFIG['n_epochs'],
    eta_min=CONFIG['lr_min']
)

# PCA + K-means ONCE on actual feature matrices (not huge raw dims)
from sklearn.decomposition import PCA
print('\n[INIT] PCA + K-means...')
t0   = time.time()
pca_u = PCA(n_components=64, random_state=42)
pca_i = PCA(n_components=64, random_state=42)
u_r   = pca_u.fit_transform(user_feat_np)
i_r   = pca_i.fit_transform(item_feat_np)
print(f'[INIT] PCA done: {time.time()-t0:.1f}s')
model.build_cluster_graph(
    torch.FloatTensor(u_r).to(DEVICE),
    torch.FloatTensor(i_r).to(DEVICE)
)

history = {
    'total':[],'BPR':[],'L1':[],
    'L2':[],'L3':[],'KL':[]
}

START_EPOCH = 0
best_ndcg   = 0.0
no_improve  = 0

print('\n✅ v3 Model ready — Early Fusion active, fresh start from epoch 0')
print('='*70)


Modality dims:   {'text': 384, 'image': 4096, 'meta': 5}
User feat dim:   4480  (for Early Fusion)
Item feat dim:   4480  (for Early Fusion)
logvar bias: tensor([-2., -2., -2.], device='cuda:0')
KL fix verified ✅

[INIT] PCA + K-means...
[INIT] PCA done: 9.9s
[MODEL] Building cluster graph (threshold=0.1)...
[MODEL] Cluster graph: 156 edges, 20u + 20i clusters ✅

✅ v3 Model ready — Early Fusion active, fresh start from epoch 0


## CELL 11 — Evaluation Functions

In [54]:
def compute_ndcg(pred_items, true_items, k):
    dcg, idcg = 0.0, 0.0
    for rank, item in enumerate(pred_items[:k]):
        if item in true_items:
            dcg += 1.0 / np.log2(rank + 2)
    for rank in range(min(len(true_items), k)):
        idcg += 1.0 / np.log2(rank + 2)
    return dcg / idcg if idcg > 0 else 0.0


@torch.no_grad()
def evaluate(model, data, config, split='test'):
    model.eval()
    k       = config['K']
    n_users = data['n_users']
    n_items = data['n_items']
    device  = config['device']

    mod_gpu = {mk: mv.to(device) for mk, mv in data['modality_features'].items()}

    # ✅ Pass user/item features for Early Fusion
    uf = data['user_features'].to(device)
    if_ = data['item_features'].to(device)
    outputs = model(data['edge_index'], mod_gpu,
                    user_features=uf, item_features=if_)
    H_U = outputs['H_U']
    H_I = outputs['H_I']

    scores = torch.matmul(H_U, H_I.T)

    # Mask training items
    for u, items in data['train_user_items'].items():
        if items:
            scores[u, items] = -1e9

    _, top_k = scores.topk(k, dim=-1)
    top_k    = top_k.cpu().numpy()

    eval_items = data['test_user_items'] if split == 'test' else data['val_user_items']

    prec, rec, ndcg, f1, acc = [], [], [], [], []
    for u in range(n_users):
        true = set(eval_items.get(u, []))
        if not true: continue
        pred = list(top_k[u])
        hits = len(set(pred) & true)
        p = hits / k
        r = hits / len(true)
        n = compute_ndcg(pred, true, k)
        f = 2*p*r/(p+r) if (p+r) > 0 else 0.0
        prec.append(p); rec.append(r)
        ndcg.append(n); f1.append(f); acc.append(p)

    # RMSE
    rmse = 0.0
    if data.get('all_ratings') is not None:
        try:
            H_U_c = H_U.cpu()
            H_I_c = H_I.cpu()
            preds, tgts = [], []
            for row in data['all_ratings'][:5000]:
                u_i = min(int(row[0]), n_users-1)
                i_i = min(int(row[1]), n_items-1)
                r   = float(row[2])
                s   = (H_U_c[u_i] * H_I_c[i_i]).sum().item()
                preds.append(1 + 4*torch.sigmoid(torch.tensor(s)).item())
                tgts.append(r)
            rmse = np.sqrt(np.mean((np.array(preds) - np.array(tgts))**2))
        except:
            rmse = 0.0

    return {
        'Precision@K': np.mean(prec) if prec else 0.0,
        'Recall@K':    np.mean(rec)  if rec  else 0.0,
        'NDCG@K':      np.mean(ndcg) if ndcg else 0.0,
        'F1-Score@K':  np.mean(f1)   if f1   else 0.0,
        'Accuracy@K':  np.mean(acc)  if acc  else 0.0,
        'RMSE':        rmse
    }


@torch.no_grad()
def evaluate_cold_start(model, data, config, split='test'):
    model.eval()
    top_k_count = config['K']
    n_users = data['n_users']
    device  = config['device']

    cold_users = [
        u for u in range(n_users)
        if len(data['train_user_items'].get(u, [])) <= config['keep_k']
    ]
    if not cold_users:
        all_u  = list(data['train_user_items'].keys())
        n_cold = max(1, int(len(all_u)*0.2))
        cold_users = all_u[:n_cold]

    mod_gpu = {mk: mv.to(device) for mk, mv in data['modality_features'].items()}
    uf  = data['user_features'].to(device)
    if_ = data['item_features'].to(device)
    outputs = model(data['edge_index'].to(device), mod_gpu,
                    user_features=uf, item_features=if_)
    H_U = outputs['H_U']
    H_I = outputs['H_I']
    scores = torch.matmul(H_U, H_I.T)
    for u, items in data['train_user_items'].items():
        if items: scores[u, items] = -1e9

    cold_t  = torch.tensor(cold_users, dtype=torch.long, device=device)
    _, top_k = scores[cold_t].topk(top_k_count, dim=-1)
    top_k   = top_k.cpu().numpy()

    eval_items = data['test_user_items'] if split=='test' else data['val_user_items']
    hits, ndcgs, recalls = [], [], []
    for i, u in enumerate(cold_users):
        true = set(eval_items.get(u, []))
        if not true: continue
        pred = list(top_k[i])
        h    = len(set(pred) & true)
        hits.append(1.0 if h > 0 else 0.0)
        recalls.append(h / len(true))
        ndcgs.append(compute_ndcg(pred, true, top_k_count))

    return {
        'ColdStart-Hit@K':    np.mean(hits)    if hits    else 0.0,
        'ColdStart-NDCG@K':   np.mean(ndcgs)   if ndcgs   else 0.0,
        'ColdStart-Recall@K': np.mean(recalls) if recalls else 0.0,
        'n_cold_users':       len(cold_users)
    }


print('Evaluation functions defined ✅')
print(f'  Users: {data["n_users"]:,} | Items: {data["n_items"]:,} | K={CONFIG["K"]}')
print('  ✅ NEW: Evaluation passes user/item features for Early Fusion inference')


Evaluation functions defined ✅
  Users: 19,445 | Items: 7,050 | K=10
  ✅ NEW: Evaluation passes user/item features for Early Fusion inference


## CELL 12 — Checkpoint Functions

In [55]:
def save_checkpoint(model, optimizer, scheduler, epoch, metrics, history, path):
    torch.save({
        'epoch':     epoch,
        'model':     model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'metrics':   metrics,
        'history':   history
    }, path)
    print(f'  [SAVED] {path}')


def load_checkpoint_resume(model, optimizer, scheduler, path):
    ckpt = torch.load(path, map_location=DEVICE)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    if 'scheduler' in ckpt:
        scheduler.load_state_dict(ckpt['scheduler'])
    print(f'[RESUMED] from epoch {ckpt["epoch"]}')
    return ckpt['epoch'], ckpt['metrics'], ckpt.get('history', {})


START_EPOCH  = 0
best_ndcg    = 0.0
no_improve   = 0
print(f'Starting fresh from epoch 0')
print(f'To resume: uncomment load_checkpoint_resume() below')
print()
print('# To resume from checkpoint:')
print('# START_EPOCH, _, history = load_checkpoint_resume(')
print('#     model, optimizer, scheduler,')
print('#     f"{PROJECT_DIR}/checkpoints/epoch_50_v2.pt"')
print('# START_EPOCH += 1')

Starting fresh from epoch 0
To resume: uncomment load_checkpoint_resume() below

# To resume from checkpoint:
# START_EPOCH, _, history = load_checkpoint_resume(
#     model, optimizer, scheduler,
#     f"{PROJECT_DIR}/checkpoints/epoch_50_v2.pt"
# START_EPOCH += 1


## CELL 13 — MAIN TRAINING LOOP (All Fixes)

In [56]:
print('=' * 70)
print('MM-CLightRec v3.0 — Amazon Baby — Early Fusion + ALL FIXES')
print('=' * 70)
print(f'λ1={CONFIG["lambda1"]} λ2={CONFIG["lambda2"]} λ3={CONFIG["lambda3"]} λ4={CONFIG["lambda4"]}')
print(f'LR: {CONFIG["lr"]} → cosine → {CONFIG["lr_min"]}')
print(f'✅ NEW: LightGCN fuses {USER_FEAT_DIM}D user + {ITEM_FEAT_DIM}D item features')
print('=' * 70)

edge_index_gpu = data['edge_index'].to(DEVICE)

for epoch in range(START_EPOCH, CONFIG['n_epochs']):
    model.train()
    t0 = time.time()
    loss_accum = dict(total=0, BPR=0, L1=0, L2=0, L3=0, KL=0)
    n_batches  = 0

    for batch in train_loader:
        batch = batch.to(DEVICE)
        u_ids, p_ids, n_ids = batch[:,0], batch[:,1], batch[:,2]

        mod_gpu = {mk: mv.to(DEVICE) for mk, mv in data['modality_features'].items()}

        # ✅ Forward with Early Fusion features
        outputs = model(
            edge_index_gpu, mod_gpu,
            user_features=user_features_t,
            item_features=item_features_t,
        )

        L_total, losses = compute_total_loss(
            outputs, model, u_ids, p_ids, n_ids, CONFIG, edge_index_gpu
        )

        optimizer.zero_grad()
        L_total.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        for k, v in losses.items():
            loss_accum[k] += v
        n_batches += 1

    scheduler.step()

    for k in loss_accum:
        loss_accum[k] /= max(n_batches, 1)
    for k in history:
        history[k].append(loss_accum[k])

    elapsed = time.time() - t0
    ep_str  = (f"Ep {epoch+1:4d}/{CONFIG['n_epochs']} | "
               f"Loss:{loss_accum['total']:.4f} | "
               f"BPR:{loss_accum['BPR']:.4f} | "
               f"L1:{loss_accum['L1']:.4f} | "
               f"L2:{loss_accum['L2']:.4f} | "
               f"L3:{loss_accum['L3']:.4f} | "
               f"KL:{loss_accum['KL']:.4f} | "
               f"lr:{scheduler.get_last_lr()[0]:.2e} | {elapsed:.0f}s")
    print(ep_str)

    if (epoch + 1) % CONFIG['eval_every'] == 0:
        val_m    = evaluate(model, data, CONFIG, split='val')
        cold_m   = evaluate_cold_start(model, data, CONFIG, split='val')
        val_ndcg = val_m['NDCG@K']
        print(f"  [VAL]  Prec:{val_m['Precision@K']:.4f} | "
              f"Rec:{val_m['Recall@K']:.4f} | "
              f"NDCG:{val_ndcg:.4f} | "
              f"F1:{val_m['F1-Score@K']:.4f} | "
              f"RMSE:{val_m['RMSE']:.4f}")
        print(f"  [COLD] Hit:{cold_m['ColdStart-Hit@K']:.4f} | "
              f"NDCG:{cold_m['ColdStart-NDCG@K']:.4f} | "
              f"Rec:{cold_m['ColdStart-Recall@K']:.4f}")

        if val_ndcg > best_ndcg:
            best_ndcg = val_ndcg
            no_improve = 0
            print(f"  [BEST] NDCG:{best_ndcg:.4f} ✅")
            ckpt_path = f"{CONFIG['checkpoint_dir']}/epoch_{epoch+1}_v3.pt"
            save_checkpoint(model, optimizer, scheduler, epoch+1, val_m, history, ckpt_path)
        else:
            no_improve += 1
            if no_improve >= CONFIG['patience'] // CONFIG['eval_every']:
                print(f"[EARLY STOP] No improvement for {CONFIG['patience']} epochs.")
                break

    if (epoch + 1) % CONFIG['save_every'] == 0:
        ckpt_path = f"{CONFIG['checkpoint_dir']}/epoch_{epoch+1}_v3.pt"
        save_checkpoint(model, optimizer, scheduler, epoch+1, {}, history, ckpt_path)

print('\n✅ Training complete')

# ── Final Test Evaluation ──────────────────────────────────────────────────
print('=' * 70)
print('FINAL EVALUATION ON TEST SET')
print('=' * 70)
test_m  = evaluate(model, data, CONFIG, split='test')
cold_m  = evaluate_cold_start(model, data, CONFIG, split='test')
print(f"{'Metric':<25} {'Value':<10}")
print('-' * 35)
for metric, value in test_m.items():
    print(f'{metric:<25} {value:<10.4f}')
print()
for metric, value in cold_m.items():
    if isinstance(value, float):
        print(f'{metric:<25} {value:<10.4f}')

# ── Plots ──────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(history['total'], label='Total Loss')
axes[0].plot(history['BPR'],   label='BPR')
axes[0].set_title('Training Loss'); axes[0].legend()

axes[1].plot(history['L1'], label='L1 Inter-Modal')
axes[1].plot(history['L2'], label='L2 Structural')
axes[1].plot(history['L3'], label='L3 Cold-Start')
axes[1].set_title('Contrastive Losses'); axes[1].legend()

metrics_val = [test_m['Precision@K'], test_m['Recall@K'],
               test_m['NDCG@K'],      test_m['F1-Score@K']]
axes[2].bar(['P@10','R@10','NDCG@10','F1@10'], metrics_val, color='steelblue')
axes[2].set_title('Final Test Metrics'); axes[2].set_ylim(0, max(metrics_val)*1.3)
for i, v in enumerate(metrics_val):
    axes[2].text(i, v + 0.001, f'{v:.4f}', ha='center', fontsize=9)

plt.tight_layout()
plot_path = f"{CONFIG['results_dir']}/plots/training_v3.png"
plt.savefig(plot_path, dpi=120)
plt.show()
print(f'[PLOT SAVED] {plot_path}')


MM-CLightRec v3.0 — Amazon Baby — Early Fusion + ALL FIXES
λ1=0.1 λ2=0.1 λ3=0.05 λ4=0.1
LR: 0.001 → cosine → 1e-05
✅ NEW: LightGCN fuses 4480D user + 4480D item features
Ep    1/300 | Loss:1.5093 | BPR:0.5802 | L1:3.0503 | L2:4.4899 | L3:3.2543 | KL:0.1242 | lr:1.00e-03 | 42s
Ep    2/300 | Loss:1.2309 | BPR:0.4950 | L1:2.1940 | L2:3.9449 | L3:2.4200 | KL:0.0100 | lr:1.00e-03 | 41s
Ep    3/300 | Loss:1.1109 | BPR:0.4458 | L1:1.9485 | L2:3.6355 | L3:2.1145 | KL:0.0100 | lr:1.00e-03 | 40s
Ep    4/300 | Loss:1.0342 | BPR:0.4115 | L1:1.8195 | L2:3.4122 | L3:1.9696 | KL:0.0100 | lr:1.00e-03 | 40s
Ep    5/300 | Loss:0.9756 | BPR:0.3808 | L1:1.7403 | L2:3.2460 | L3:1.9044 | KL:0.0100 | lr:9.99e-04 | 41s
Ep    6/300 | Loss:0.9312 | BPR:0.3552 | L1:1.6904 | L2:3.1362 | L3:1.8473 | KL:0.0100 | lr:9.99e-04 | 40s
Ep    7/300 | Loss:0.8927 | BPR:0.3327 | L1:1.6599 | L2:3.0256 | L3:1.8089 | KL:0.0100 | lr:9.99e-04 | 40s
Ep    8/300 | Loss:0.8580 | BPR:0.3124 | L1:1.6315 | L2:2.9177 | L3:1.7936 | KL:0

In [57]:
import json

# Load best model
best_path = f'{PROJECT_DIR}/checkpoints/best_model_v2.pt'
if os.path.exists(best_path):
    model.load_state_dict(
        torch.load(best_path, map_location=DEVICE)
    )
    print('Best model loaded ✅')

# Final test evaluation
print('\n' + '='*60)
print('FINAL TEST EVALUATION — MM-CLightRec v2.0')
print('='*60)

test_m = evaluate(model, data, CONFIG, split='test')
cold_m = evaluate_cold_start(model, data, CONFIG, split='test')

print(f'\n{"Metric":<22} {"Value":>10}')
print('-'*34)
for k, v in test_m.items():
    print(f'{k:<22} {v:>10.4f}')

print(f'\n{"Cold Metric":<25} {"Value":>10}')
print('-'*37)
for k, v in cold_m.items():
    if k != 'n_cold_users':
        print(f'{k:<25} {v:>10.4f}')
print(f'{"Cold users":<25} {cold_m["n_cold_users"]:>10}')

# Save
results = {
    'model':   'MM-CLightRec-v2',
    'dataset': 'Amazon Baby',
    'standard_metrics':   test_m,
    'cold_start_metrics': cold_m,
}
with open(f'{PROJECT_DIR}/results/FINAL_TEST_v2.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nSaved ✅')



FINAL TEST EVALUATION — MM-CLightRec v2.0

Metric                      Value
----------------------------------
Precision@K                0.0029
Recall@K                   0.0189
NDCG@K                     0.0114
F1-Score@K                 0.0049
Accuracy@K                 0.0029
RMSE                       1.3077

Cold Metric                    Value
-------------------------------------
ColdStart-Hit@K               0.0210
ColdStart-NDCG@K              0.0091
ColdStart-Recall@K            0.0160
Cold users                     10304

Saved ✅


In [58]:
import os

# ── Step 1: Check what checkpoints are available ────
ckpt_dir = f'{PROJECT_DIR}/checkpoints'
ckpts = sorted([f for f in os.listdir(ckpt_dir)
                if 'v2' in f and f.endswith('.pt')])
print('Available checkpoints:')
for c in ckpts:
    size = os.path.getsize(f'{ckpt_dir}/{c}') / 1e6
    print(f'  {c}  ({size:.1f} MB)')

Available checkpoints:


In [59]:
# ── Step 2: Load latest checkpoint ──────────────────
# Change epoch number to your latest saved checkpoint
# e.g. if you saved every 10 epochs and stopped at 182
# → use epoch_180_v2.pt

RESUME_EPOCH = 180   # ← change to your latest checkpoint

ckpt_path = f'{PROJECT_DIR}/checkpoints/epoch_{RESUME_EPOCH}_v2.pt'

if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    if 'scheduler' in ckpt:
        scheduler.load_state_dict(ckpt['scheduler'])
    history    = ckpt.get('history', history)
    best_ndcg  = max(history.get('total', [0])) if history.get('total') else 0.0

    # Recalculate best_ndcg from history
    # Run quick val eval to get current best
    print(f'Checkpoint loaded from epoch {RESUME_EPOCH} ✅')
    print(f'Resuming from epoch {RESUME_EPOCH + 1}')
else:
    print(f'File not found: {ckpt_path}')
    print('Check available files above and update RESUME_EPOCH')

# ── Step 3: Re-evaluate to get best_ndcg ────────────
print('\nRunning quick val evaluation to restore best_ndcg...')
val_check = evaluate(model, data, CONFIG, split='val')
best_ndcg = val_check['NDCG@K']
no_improve = 0
START_EPOCH = RESUME_EPOCH  # training continues from here

print(f'Current val NDCG: {best_ndcg:.4f}')
print(f'Will train from epoch {START_EPOCH + 1} → {CONFIG["n_epochs"]}')

File not found: /content/drive/MyDrive/MM_CLightRec_Baby_v2/checkpoints/epoch_180_v2.pt
Check available files above and update RESUME_EPOCH

Running quick val evaluation to restore best_ndcg...
Current val NDCG: 0.0120
Will train from epoch 181 → 300


In [60]:
# ── PATCH 1: Fix KL collapse ─────────────────────────
# Increase KL weight so it cannot be ignored
# Detach KL from main gradient occasionally

def vgae_kl_loss(mu, logvar):
    logvar = logvar.clamp(-4, 4)
    # Add small epsilon to prevent exact collapse
    kl = -0.5 * torch.mean(
        1 + logvar - mu.pow(2) - logvar.exp()
    )
    # Force minimum KL (prevents collapse to 0)
    return torch.clamp(kl, min=0.01)

# ── PATCH 2: Fix L2 — stronger augmentation ─────────
def compute_L2_structural(model, edge_index, config, device):
    """
    Stronger augmentation:
    30% edge dropout (was 10%)
    + Node feature noise
    """
    n_edges = edge_index.size(1)

    # View 1: drop 30% of edges
    mask1    = torch.rand(n_edges) > 0.3  # ← 0.1 to 0.3
    ei_view1 = edge_index[:, mask1].to(device)

    # View 2: drop DIFFERENT 30% + flip some
    mask2    = torch.rand(n_edges) > 0.3
    ei_view2 = edge_index[:, mask2].to(device)

    # Propagate on both views
    H_U1, H_I1 = model.lightgcn(ei_view1)
    H_U2, H_I2 = model.lightgcn(ei_view2)

    # Add small noise to one view (stronger contrast)
    H_U2 = H_U2 + torch.randn_like(H_U2) * 0.01

    z1 = F.normalize(H_U1, dim=-1)
    z2 = F.normalize(H_U2, dim=-1)

    idx = torch.randperm(z1.size(0), device=device)[:256]
    return info_nce_loss(z1[idx], z2[idx], config['temperature'])


# ── PATCH 3: Increase lambda4 ───────────────────────
CONFIG['lambda4'] = 0.5   # ← increase from 0.1 to 0.5
                           # Forces KL to stay non-zero

print('Patches applied:')
print(f'  KL: min value clamped to 0.01')
print(f'  L2: edge dropout 10% → 30%')
print(f'  lambda4: 0.1 → 0.5')

Patches applied:
  KL: min value clamped to 0.01
  L2: edge dropout 10% → 30%
  lambda4: 0.1 → 0.5


In [ ]:
# Load best checkpoint so far (epoch 20)
model.load_state_dict(
    torch.load(
        f'{PROJECT_DIR}/checkpoints/epoch_20_v2.pt',
        map_location=DEVICE
    )['model']
)
print('Resumed from epoch 20 ✅')
print('Continuing with patches applied')

# Reset tracking
best_ndcg  = 0.0123  # keep current best
no_improve = 0
START_EPOCH = 20


## CELL 14 — Final Test Evaluation

In [61]:
import json

# Load best model
best_path = f'{PROJECT_DIR}/checkpoints/best_model_v2.pt'
if os.path.exists(best_path):
    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    print('Best model loaded ✅')

print('\n' + '='*70)
print('FINAL EVALUATION — TEST SET')
print('='*70)

test_m  = evaluate(model, data, CONFIG, split='test')
cold_m  = evaluate_cold_start(model, data, CONFIG, split='test')

print(f'\n{"Metric":<22} {"Value":>10}')
print('-'*34)
for k, v in test_m.items():
    print(f'{k:<22} {v:>10.4f}')

print('\n' + '='*70)
print('COLD-START METRICS')
print('='*70)
for k, v in cold_m.items():
    if k != 'n_cold_users':
        print(f'{k:<25} {v:>10.4f}')
print(f'{"Cold-start users":<25} {cold_m["n_cold_users"]:>10}')

print('\n' + '='*70)
print('PAPER RESULTS — SCI-2026')
print('='*70)
print(f'Model:      MM-CLightRec v2.0')
print(f'Dataset:    Amazon Baby')
print(f'K:          {CONFIG["K"]}')
print(f'Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

# Save
results = {
    'model':   'MM-CLightRec-v2',
    'dataset': 'Amazon Baby',
    'version': 'v2_all_fixes',
    'standard_metrics':     test_m,
    'cold_start_metrics':   cold_m,
    'config': {k: str(v) for k, v in CONFIG.items() if k != 'device'}
}
save_path = f'{PROJECT_DIR}/results/FINAL_v2.json'
with open(save_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nResults saved: {save_path} ✅')


FINAL EVALUATION — TEST SET

Metric                      Value
----------------------------------
Precision@K                0.0029
Recall@K                   0.0189
NDCG@K                     0.0114
F1-Score@K                 0.0049
Accuracy@K                 0.0029
RMSE                       1.3077

COLD-START METRICS
ColdStart-Hit@K               0.0210
ColdStart-NDCG@K              0.0091
ColdStart-Recall@K            0.0160
Cold-start users               10304

PAPER RESULTS — SCI-2026
Model:      MM-CLightRec v2.0
Dataset:    Amazon Baby
K:          10
Parameters: 3,790,019

Results saved: /content/drive/MyDrive/MM_CLightRec_Baby_v2/results/FINAL_v2.json ✅


## CELL 15 — Conference-Quality Plots

In [62]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

plt.rcParams.update({
    'font.family':       'DejaVu Sans',
    'font.size':         12,
    'axes.titlesize':    14,
    'axes.labelsize':    12,
    'axes.titleweight':  'bold',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'figure.dpi':        300,
    'savefig.dpi':       300,
    'savefig.bbox':      'tight',
    'savefig.facecolor': 'white',
})

PLOT_DIR = f'{PROJECT_DIR}/results/plots'
os.makedirs(PLOT_DIR, exist_ok=True)
epochs = list(range(1, len(history['total'])+1))

# ── PLOT 1: All losses combined ──────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
colors  = {'total':'#1565C0','BPR':'#2E7D32','L1':'#E65100',
           'L2':'#AD1457','L3':'#6A1B9A','KL':'#00695C'}
styles  = {'total':'-','BPR':'--','L1':'--','L2':':','L3':':','KL':'-.'}
widths  = {'total':2.5,'BPR':1.5,'L1':1.5,'L2':1.5,'L3':1.5,'KL':1.5}
for name in ['total','BPR','L1','L2','L3','KL']:
    if history[name]:
        ax.plot(epochs, history[name],
                color=colors[name], linestyle=styles[name],
                linewidth=widths[name], label=f'L_{name}', alpha=0.9)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('MM-CLightRec v2.0 Loss Convergence\nAmazon Baby Dataset')
ax.legend(loc='upper right', framealpha=0.9, ncol=2)
ax.grid(True, alpha=0.2, linestyle='--')
ax.set_facecolor('#FAFAFA')
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/01_all_losses.png')
plt.show(); plt.close()
print('✅ 01_all_losses.png')

# ── PLOT 2: Individual losses (2x3 grid) ─────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('MM-CLightRec v2.0 — Individual Loss Curves', fontsize=14, fontweight='bold')
loss_info = [
    ('total','Total Loss','#1565C0'),
    ('BPR','BPR Ranking Loss','#2E7D32'),
    ('L1','L₁ Inter-Modal Contrastive','#E65100'),
    ('L2','L₂ Structural Contrastive (Fixed)','#AD1457'),
    ('L3','L₃ Cold-Start Contrastive','#6A1B9A'),
    ('KL','KL Divergence (Fixed)','#00695C'),
]
for idx, (name, title, color) in enumerate(loss_info):
    ax = axes[idx//3][idx%3]
    if history[name]:
        ax.plot(epochs, history[name], color=color, linewidth=2)
        ax.fill_between(epochs, history[name], alpha=0.07, color=color)
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.set_xlabel('Epoch', fontsize=10)
    ax.set_ylabel('Loss', fontsize=10)
    ax.grid(True, alpha=0.2, linestyle='--')
    ax.set_facecolor('#FAFAFA')
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/02_individual_losses.png')
plt.show(); plt.close()
print('✅ 02_individual_losses.png')

# ── PLOT 3: Standard metrics bar chart ───────────────
mnames = ['Precision\n@10','Recall\n@10','NDCG\n@10','F1\n@10','Accuracy\n@10']
mvals  = [test_m['Precision@K'],test_m['Recall@K'],test_m['NDCG@K'],
          test_m['F1-Score@K'],test_m['Accuracy@K']]
mcolors= ['#1565C0','#2E7D32','#E65100','#AD1457','#00838F']
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(mnames, mvals, color=mcolors, alpha=0.88, width=0.55, edgecolor='white')
for bar, val in zip(bars, mvals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.0004,
            f'{val:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_ylim(0, max(mvals)*1.35+0.01)
ax.set_ylabel('Score')
ax.set_title('MM-CLightRec v2.0 — Standard Metrics (K=10)\nAmazon Baby Test Set')
ax.grid(True, alpha=0.2, axis='y', linestyle='--')
ax.set_facecolor('#FAFAFA')
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/03_standard_metrics.png')
plt.show(); plt.close()
print('✅ 03_standard_metrics.png')

# ── PLOT 4: Cold-start metrics bar chart ─────────────
cnames = ['ColdStart\nHit@10','ColdStart\nNDCG@10','ColdStart\nRecall@10']
cvals  = [cold_m['ColdStart-Hit@K'],cold_m['ColdStart-NDCG@K'],cold_m['ColdStart-Recall@K']]
ccolors= ['#4527A0','#6A1B9A','#880E4F']
fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(cnames, cvals, color=ccolors, alpha=0.88, width=0.45, edgecolor='white')
for bar, val in zip(bars, cvals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.0003,
            f'{val:.4f}', ha='center', va='bottom', fontsize=12, fontweight='bold')
ax.set_ylim(0, max(cvals)*1.4+0.005)
ax.set_ylabel('Score')
ax.set_title(f'MM-CLightRec v2.0 — Cold-Start Evaluation\n'
             f'({cold_m["n_cold_users"]:,} Cold Users, K=5 shots, K=10 rec)')
ax.grid(True, alpha=0.2, axis='y', linestyle='--')
ax.set_facecolor('#FAFAFA')
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/04_cold_start_metrics.png')
plt.show(); plt.close()
print('✅ 04_cold_start_metrics.png')

print(f'\nAll plots saved to: {PLOT_DIR}')
print('Use plots 01, 03, 04 in your paper ✅')

✅ 01_all_losses.png
✅ 02_individual_losses.png
✅ 03_standard_metrics.png
✅ 04_cold_start_metrics.png

All plots saved to: /content/drive/MyDrive/MM_CLightRec_Baby_v2/results/plots
Use plots 01, 03, 04 in your paper ✅


## CELL 16 — Paper Results Summary

In [ ]:
print('='*65)
print('PAPER RESULTS SUMMARY — SCI-2026 (Springer LNNS)')
print('='*65)
print(f'Model:      MM-CLightRec v2.0')
print(f'Dataset:    Amazon Baby')
print(f'K=10 | λ1={CONFIG["lambda1"]} λ2={CONFIG["lambda2"]} λ3={CONFIG["lambda3"]} λ4={CONFIG["lambda4"]}')
print()
print('Table 1: Standard Recommendation Metrics')
print(f'{"Metric":<20} {"MM-CLightRec":>14}')
print('-'*35)
for k, v in test_m.items():
    print(f'{k:<20} {v:>14.4f}')
print()
print('Table 2: Cold-Start Metrics (Exclusive to Our Model)')
print(f'{"Metric":<25} {"MM-CLightRec":>14}')
print('-'*40)
for k, v in cold_m.items():
    if k != 'n_cold_users':
        print(f'{k:<25} {v:>14.4f}')
print(f'{"Cold-start users":<25} {cold_m["n_cold_users"]:>14}')
print()
print('='*65)
print('MODEL EFFICIENCY')
print(f'  Parameters:    {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
print(f'  Base paper:    ~32,000,000')
print(f'  Reduction:     ~{(1-sum(p.numel() for p in model.parameters() if p.requires_grad)/32e6)*100:.1f}%')
print()
print('LOSS CONVERGENCE')
if history['BPR']:
    print(f'  BPR:  {history["BPR"][0]:.4f} → {history["BPR"][-1]:.4f}')
    print(f'  L1:   {history["L1"][0]:.4f} → {history["L1"][-1]:.4f}')
    print(f'  L2:   {history["L2"][0]:.4f} → {history["L2"][-1]:.4f}')
    print(f'  L3:   {history["L3"][0]:.4f} → {history["L3"][-1]:.4f}')
    print(f'  KL:   {history["KL"][0]:.4f} → {history["KL"][-1]:.4f}')
print('='*65)

In [63]:
# ════════════════════════════════════════════════════════════════════════
# SAMPLED NEGATIVE EVALUATION (Matches Base Paper Protocol)
# 1 positive + 99 negatives per user (no retraining needed)
# ════════════════════════════════════════════════════════════════════════

print('\n' + '='*70)
print('  RUNNING SAMPLED EVALUATION (1 Pos + 99 Neg) on Best Saved Model')
print('='*70)

N_NEGATIVE = 99
K = 10
SEED = 42

# 1. Load best model weights
best_ckpt = f"{CONFIG['checkpoint_dir']}/epoch_{epoch+1}_v3.pt"
import glob
checkpoints = glob.glob(f"{CONFIG['checkpoint_dir']}/epoch_*_v3.pt")
if checkpoints:
    # Get the latest checkpoint
    checkpoints.sort(key=os.path.getmtime)
    best_ckpt = checkpoints[-1]
    state = torch.load(best_ckpt, map_location=DEVICE)
    if 'model_state_dict' in state:
        model.load_state_dict(state['model_state_dict'])
    else:
        model.load_state_dict(state)
    print(f"[INFO] Loaded model from {best_ckpt} ✅")
else:
    print("[WARNING] No checkpoints found, running on whatever is currently in VRAM.")

model.eval()

# 2. Compute full score matrix efficiently
print("[INFO] Computing full (user x item) score matrix...")
with torch.no_grad():
    mod_gpu = {mk: mv.to(DEVICE) for mk, mv in data['modality_features'].items()}
    uf = data['user_features'].to(DEVICE)
    if_ = data['item_features'].to(DEVICE)

    # Forward pass to get embeddings
    outputs = model(data['edge_index'].to(DEVICE), mod_gpu,
                    user_features=uf, item_features=if_)

    H_U = outputs['H_U']
    H_I = outputs['H_I']

    # Full dot product matrix
    scores = torch.matmul(H_U, H_I.T).cpu().numpy()

print(f"[INFO] Score matrix: {scores.shape} ✅")

# 3. Sampled Evaluation Engine
def compute_ndcg_sampled(ranked_list, pos_item, k):
    for rank, item in enumerate(ranked_list[:k]):
        if item == pos_item:
            return 1.0 / np.log2(rank + 2)
    return 0.0

rng = np.random.RandomState(SEED)
prec_list, rec_list, ndcg_list, f1_list = [], [], [], []

n_items = data['n_items']
for u, pos_items in data['test_user_items'].items():
    if not pos_items: continue

    pos_item = pos_items[0] # Take first positive

    # All items user has ever seen
    seen = set(data['train_user_items'].get(u, [])) | set(pos_items)

    # Sample 99 unknown items
    unseen = np.array([i for i in range(n_items) if i not in seen])
    negs = rng.choice(unseen, size=min(n_neg, len(unseen)), replace=False) if len(unseen) > N_NEGATIVE else unseen

    candidates = np.concatenate([[pos_item], negs])

    # Score candidate pool
    s = scores[u][candidates]
    order = np.argsort(-s)
    ranked = candidates[order]

    # Metrics
    hits = int(pos_item in ranked[:K])
    p = hits / K
    r = float(hits)  # Recall is hits/1 since only 1 pos item
    nd = compute_ndcg_sampled(ranked, pos_item, K)
    f = 2*p*r/(p+r) if (p+r) > 0 else 0.0

    prec_list.append(p)
    rec_list.append(r)
    ndcg_list.append(nd)
    f1_list.append(f)

print('\n' + '='*70)
print(f'          SAMPLED EVALUATION RESULTS (K={K})')
print('='*70)
print(f'  Precision@{K}  : {np.mean(prec_list):.4f}')
print(f'  Recall@{K}     : {np.mean(rec_list):.4f}')
print(f'  NDCG@{K}       : {np.mean(ndcg_list):.4f}')
print(f'  F1-Score@{K}   : {np.mean(f1_list):.4f}')
print('='*70)
print('  (These numbers are directly comparable to the MGRS-HFA paper)')
print('='*70)


  RUNNING SAMPLED EVALUATION (1 Pos + 99 Neg) on Best Saved Model


RuntimeError: Error(s) in loading state_dict for MM_CLightRec:
	Missing key(s) in state_dict: "lightgcn.user_emb.weight", "lightgcn.item_emb.weight", "lightgcn.user_feat_proj.weight", "lightgcn.user_feat_proj.bias", "lightgcn.item_feat_proj.weight", "lightgcn.item_feat_proj.bias", "text_lgcn.proj.weight", "text_lgcn.proj.bias", "image_lgcn.proj.weight", "image_lgcn.proj.bias", "meta_lgcn.proj.weight", "meta_lgcn.proj.bias", "fusion_w.weight", "fusion_w.bias", "proj_text.net.0.weight", "proj_text.net.0.bias", "proj_text.net.1.weight", "proj_text.net.1.bias", "proj_text.net.1.running_mean", "proj_text.net.1.running_var", "proj_text.net.4.weight", "proj_text.net.4.bias", "proj_image.net.0.weight", "proj_image.net.0.bias", "proj_image.net.1.weight", "proj_image.net.1.bias", "proj_image.net.1.running_mean", "proj_image.net.1.running_var", "proj_image.net.4.weight", "proj_image.net.4.bias", "W_Q.weight", "W_Q.bias", "W_K.weight", "W_K.bias", "W_V.weight", "W_V.bias", "vgae_mu.weight", "vgae_